# Four-stage compressor digital twin: end-to-end walkthrough
Team 12 (AE 541). This notebook runs the whole chain on simulated data: physics model → dataset → POD reduced-order model → twin loop.
**All numbers here are model predictions (simulated), not measurements.** Run it from the `notebooks/` folder.

In [ ]:
import os, sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO = os.path.abspath('..')
for sub in ('modelling', 'twin'):
    sys.path.insert(0, os.path.join(REPO, 'src', sub))
import offdesign as od, rom_eval, compressor_rpm_calculator as cc
from twin_core import Twin

## 1. Full-order model at the design point (20,000 rpm)

In [ ]:
C = od.Compressor('B')
op = C.operating_point(20000.0)
print(f"PR = {op['PR']:.4f},  T_out = {op['T0']:.2f} K,  mdot = {op['m']:.4f} kg/s,  eta = {op['eta']:.3f},  SM = {op['surge_margin']*100:.1f} %")

## 2. Change the speed and recompute everything (same as `src/modelling/compressor_rpm_calculator.py`)

In [ ]:
r = cc.calculate(15000)          # <- change the rpm here
cc.print_report(r)

## 3. Simulated dataset and compressor map

In [ ]:
df = pd.read_csv(os.path.join(REPO, 'data', 'processed', 'Compressor_4stage_dataset.csv'))
mp = pd.read_csv(os.path.join(REPO, 'data', 'processed', 'compressor_map_speedlines.csv'))
su = pd.read_csv(os.path.join(REPO, 'data', 'processed', 'surge_line.csv'))
fig, ax = plt.subplots(figsize=(6, 4))
for N, g in mp.groupby('rpm'):
    ax.plot(g.mdot_kg_s, g.PR, lw=0.8, color='0.5')
ax.plot(su.surge_mdot_kg_s, su.surge_PR, 'r--', label='surge line')
ax.plot(df.mdot_kg_s, df.PR, 'o', ms=3, label='dataset (valve fully open)')
ax.set_xlabel('mass flow, kg/s'); ax.set_ylabel('total pressure ratio'); ax.legend(); ax.set_title('SIMULATED compressor map')
plt.show()

## 4. POD reduced-order model vs full-order model on the 15 held-out points

In [ ]:
te = pd.read_csv(os.path.join(REPO, 'models', 'pod_rom', 'doe_test_15.csv'))
pred = np.array([rom_eval.predict(r.N, r.phi)['PR'] for r in te.itertuples()])
err = pred - te.PR.values
print(f'max |PR error| = {abs(err).max():.2e},  RMSE = {np.sqrt((err**2).mean()):.2e}')
R = json.load(open(os.path.join(REPO, 'models', 'pod_rom', 'rom_pod.json')))
print('retained modes r =', R['r'], ' energy =', R['energy'][R['r'] - 1])
plt.figure(figsize=(4, 4)); plt.plot(te.PR, pred, 'o'); plt.plot([1, 1.13], [1, 1.13], 'k:')
plt.xlabel('full-order model'); plt.ylabel('ROM'); plt.title('Pressure ratio (test points)'); plt.show()

## 5. Twin loop: hold PR = 1.10, then a 10 % motor-speed sag

In [ ]:
tw = Twin(None); tw.set(mode='AUTO', PR_set=1.10, sag=0.0, running=True)
t, pr, n = [], [], []
for k in range(600):
    if k == 300: tw.set(sag=0.10)
    tw.step(); s = tw.last; t.append(s['t']); pr.append(s['PR_meas']); n.append(s['rpm'])
fig, ax = plt.subplots(2, 1, figsize=(6, 4), sharex=True)
ax[0].plot(t, pr); ax[0].axhline(1.10, ls=':', c='k'); ax[0].set_ylabel('PR (simulated)')
ax[1].plot(t, n); ax[1].set_ylabel('rpm'); ax[1].set_xlabel('t, s'); plt.show()